# 03 — House PDF quality & smoke test

## Objectif

Mesurer la qualité technique des PDF House téléchargés.

Produire :
- `house_pdf_text_quality.csv` ;
- `house_sample_extraction_smoke_test.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.pdf_quality import (
    run_pdf_quality_audit, summarize_pdf_quality, choose_smoke_sample,
    run_smoke_test, append_quality_report,
)
from src.utils import AUDIT_DIR, require_file

print("Imports OK")

Imports OK


In [2]:
MAX_FILES = None  # Mettre 50 pour un audit rapide.
manifest_path = AUDIT_DIR / "house_pdf_manifest.csv"
quality_path = AUDIT_DIR / "house_pdf_text_quality.csv"
smoke_path = AUDIT_DIR / "house_sample_extraction_smoke_test.csv"

In [3]:
require_file(manifest_path, "Run 02_house_pdf_download_manifest.ipynb first.")
manifest = pd.read_csv(manifest_path, dtype={"doc_id": str})

print("manifest shape:", manifest.shape)
display(manifest["download_status"].value_counts(dropna=False))
display(manifest[["year", "n_pages"]].describe())

manifest shape: (515, 16)


download_status
ok    515
Name: count, dtype: int64

,year,n_pages
count,515.0,515.000000
mean,2025.0,3.541748
std,0.0,6.715429
min,2025.0,1.000000
25%,2025.0,1.000000
50%,2025.0,2.000000
75%,2025.0,3.000000
max,2025.0,81.000000


In [4]:
quality = run_pdf_quality_audit(
    manifest,
    output_path=quality_path,
    max_files=MAX_FILES,
)

print("quality shape:", quality.shape)
display(quality.head())

PDF quality:   0%|          | 0/515 [00:00<?, ?it/s]

quality shape: (515, 12)


,year,doc_id,local_path,n_pages,text_chars_first_pages,contains_periodic_transaction_report,contains_filer_information,contains_transactions,contains_amount,text_extractable_flag,quality_status,error_message
0,2025,20032062,/Users/lemairealice/Downloads/Jupiter/congress...,1,925,True,False,True,True,True,ok_text,
1,2025,20026537,/Users/lemairealice/Downloads/Jupiter/congress...,2,1640,True,False,True,True,True,ok_text,
2,2025,20026727,/Users/lemairealice/Downloads/Jupiter/congress...,2,1668,True,False,True,True,True,ok_text,
3,2025,20027927,/Users/lemairealice/Downloads/Jupiter/congress...,1,1015,True,False,True,True,True,ok_text,
4,2025,20030316,/Users/lemairealice/Downloads/Jupiter/congress...,2,1848,True,False,True,True,True,ok_text,


In [5]:
summary = summarize_pdf_quality(quality)
summary

{'timestamp': '2026-06-23T16:08:51+00:00',
 'n_pdf_audited': 515,
 'status_counts': {'ok_text': 449, 'no_text': 66},
 'ok_text_rate': 0.8718446601941747,
 'weak_text_rate': 0.0,
 'no_text_rate': 0.12815533980582525,
 'unreadable_rate': 0.0,
 'n_pages_distribution': {'count': 515.0,
  'mean': 3.541747572815534,
  'std': 6.715429186429295,
  'min': 1.0,
  '25%': 1.0,
  '50%': 2.0,
  '75%': 3.0,
  'max': 81.0}}

In [6]:
display(quality["quality_status"].value_counts(dropna=False))
problem_cols = ["year", "doc_id", "quality_status", "text_chars_first_pages", "error_message"]
display(quality[quality["quality_status"] != "ok_text"][problem_cols].head(20))

quality_status
ok_text    449
no_text     66
Name: count, dtype: int64

,year,doc_id,quality_status,text_chars_first_pages,error_message
27,2025,8220747,no_text,1,
146,2025,8220753,no_text,1,
147,2025,8220902,no_text,1,
148,2025,9115670,no_text,1,
149,2025,8221237,no_text,1,
176,2025,8220796,no_text,0,
177,2025,8220809,no_text,0,
178,2025,8220845,no_text,0,
284,2025,8220750,no_text,1,
285,2025,8220783,no_text,1,


In [7]:
sample = choose_smoke_sample(manifest, quality, random_state=42)
print("sample shape:", sample.shape)
display(sample[["year", "doc_id", "declarant_name_raw", "n_pages"]].head())

sample shape: (47, 16)


,year,doc_id,declarant_name_raw,n_pages
0,2025,20032062,Robert B. Aderholt,1
10,2025,20033415,Richard W. Allen,1
27,2025,8220747,Gus M. Bilirakis,2
30,2025,20027940,Rob Bresnahan,2
78,2025,20030692,Michael A. Collins,1


In [8]:
smoke = run_smoke_test(sample, output_path=smoke_path)
print("smoke shape:", smoke.shape)
display(smoke.head())

Smoke regex:   0%|          | 0/47 [00:00<?, ?it/s]

smoke shape: (47, 9)


,doc_id,year,declarant_name_raw,state_dst,text_chars,tickers_seen_raw_regex,transaction_dates_seen_regex,amount_ranges_seen_regex,notes
0,20032062,2025,Robert B. Aderholt,AL04,925,GSK,09/10/2025,"$1,001 - $15,000|>\n$200",
1,20033415,2025,Richard W. Allen,GA12,1161,CHD,11/13/2025,"$15,001 -\n$50,000|$50,001 -\n$100,000|>\n$200",
2,8220747,2025,Gus M. Bilirakis,FL12,1,,,,
3,20027940,2025,Rob Bresnahan,PA08,2243,ALAB|BABA|HOOD|INTC|META|ORCL|RDDT,03/10/2025,"$1,001 - $15,000|>\n$200",
4,20030692,2025,Michael A. Collins,GA10,914,,07/19/2025,"$1,001 - $15,000|>\n$200",


In [9]:
report_path = append_quality_report(summary)
print("Written:", quality_path.relative_to(ROOT))
print("Written:", smoke_path.relative_to(ROOT))
print("Updated:", report_path.relative_to(ROOT))

Written: data/audit/house_pdf_text_quality.csv
Written: data/audit/house_sample_extraction_smoke_test.csv
Updated: reports/house_download_audit.md


## Conclusion

**OK** si les PDF sont majoritairement `ok_text` et que les cas faibles sont listés.

**NEXT** — lancer le diagnostic Quiver avec `04_quiver_access_validation_2025.ipynb`.